In [1]:
import pickle

with open('df_cleaned.pkl', 'rb') as f:
    df = pickle.load(f)

print("Loaded!", df.shape)

Loaded! (7032, 31)


In [2]:
with open('rf_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

with open('X_train.pkl', 'rb') as f:
    X_train = pickle.load(f)

with open('X_test.pkl', 'rb') as f:
    X_test = pickle.load(f)

with open('y_train.pkl', 'rb') as f:
    y_train = pickle.load(f)

with open('y_test.pkl', 'rb') as f:
    y_test = pickle.load(f)

print("All loaded!")

All loaded!


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

sample_sizes = [0.1, 0.25, 0.5, 0.75, 1.0]
results = []

for size in sample_sizes:
    n_samples = int(len(X_train) * size)
    X_subset = X_train.iloc[:n_samples]
    y_subset = y_train.iloc[:n_samples]

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_subset, y_subset)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    results.append((size, n_samples, acc))
    print(f"Training size: {int(size*100)}% ({n_samples} rows) -> Accuracy: {acc:.4f}")

Training size: 10% (562 rows) -> Accuracy: 0.7846
Training size: 25% (1406 rows) -> Accuracy: 0.7804
Training size: 50% (2812 rows) -> Accuracy: 0.7818
Training size: 75% (4218 rows) -> Accuracy: 0.7861
Training size: 100% (5625 rows) -> Accuracy: 0.7854


In [4]:
from sklearn.model_selection import train_test_split

sample_sizes = [0.1, 0.25, 0.5, 0.75, 1.0]
n_runs = 5

all_results = {size: [] for size in sample_sizes}

for run in range(n_runs):
    for size in sample_sizes:
        n_samples = int(len(X_train) * size)

        X_subset = X_train.sample(n=n_samples, random_state=run)
        y_subset = y_train.loc[X_subset.index]

        model = RandomForestClassifier(n_estimators=100, random_state=42)
        model.fit(X_subset, y_subset)

        preds = model.predict(X_test)
        acc = accuracy_score(y_test, preds)

        all_results[size].append(acc)

print("Average accuracy across 5 runs per training size:\n")
for size in sample_sizes:
    accs = all_results[size]
    avg = sum(accs) / len(accs)
    print(f"Training size: {int(size*100)}% -> Avg Accuracy: {avg:.4f} (runs: {[round(a,4) for a in accs]})")

Average accuracy across 5 runs per training size:

Training size: 10% -> Avg Accuracy: 0.7768 (runs: [0.7676, 0.7747, 0.7882, 0.774, 0.7797])
Training size: 25% -> Avg Accuracy: 0.7777 (runs: [0.7697, 0.7854, 0.7726, 0.7861, 0.7747])
Training size: 50% -> Avg Accuracy: 0.7861 (runs: [0.7896, 0.7939, 0.7804, 0.7839, 0.7825])
Training size: 75% -> Avg Accuracy: 0.7810 (runs: [0.7868, 0.7804, 0.7818, 0.7846, 0.7711])
Training size: 100% -> Avg Accuracy: 0.7858 (runs: [0.7854, 0.7889, 0.7861, 0.7882, 0.7804])


In [5]:
model_balanced = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model_balanced.fit(X_train, y_train)

preds_balanced = model_balanced.predict(X_test)

print("Balanced Model Accuracy:", accuracy_score(y_test, preds_balanced))
print(classification_report(y_test, preds_balanced))

Balanced Model Accuracy: 0.7839374555792467


NameError: name 'classification_report' is not defined

In [6]:
from sklearn.metrics import accuracy_score, classification_report

In [7]:
print("Balanced Model Accuracy:", accuracy_score(y_test, preds_balanced))
print(classification_report(y_test, preds_balanced))

Balanced Model Accuracy: 0.7839374555792467
              precision    recall  f1-score   support

           0       0.82      0.91      0.86      1033
           1       0.63      0.45      0.53       374

    accuracy                           0.78      1407
   macro avg       0.73      0.68      0.69      1407
weighted avg       0.77      0.78      0.77      1407



In [8]:
probs = rf_model.predict_proba(X_test)[:, 1]

threshold = 0.35
preds_threshold = (probs >= threshold).astype(int)

print(f"Threshold: {threshold}")
print("Accuracy:", accuracy_score(y_test, preds_threshold))
print(classification_report(y_test, preds_threshold))

Threshold: 0.35
Accuracy: 0.7611940298507462
              precision    recall  f1-score   support

           0       0.87      0.79      0.83      1033
           1       0.54      0.68      0.60       374

    accuracy                           0.76      1407
   macro avg       0.71      0.74      0.72      1407
weighted avg       0.78      0.76      0.77      1407

